# User-Level Sockpuppet Classification on Karin's Pantip Data

**Task**: Binary classification of individual users as sockpuppet (1) or normal (0).

**Comparison**:
- **Set A — Karin-style features (aggregated to user-level)**: forum entropy, unique-words self-similarity, post-time distribution, etc.
- **Set B — Lindsey-style behavioral features**: posting rhythm, activity patterns, content statistics aggregated from individual posts.
- **Set C — Combined**: A + B.

**Key design decisions**:
- Unit of analysis: **user** (not post).
- Train/test split: stratified by `sockpuppet_group_id` to prevent group leakage.
- Cross-validation: GroupKFold by `sockpuppet_group_id` (users sharing IPs stay together).
- No GABR. No community detection. No group identification.

In [1]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

## 1. Load preprocessed post-level data

Expects the output of `00_new_preprocessing_karin.ipynb` with columns:
`thread_id, topic, num_0, num_1, user_id, messages, length, datetime, user_react, is_sockpuppet, sockpuppet_group_id, tag` (and optionally `forum`).

In [2]:
DATA_PATH = '00_Datasets_After_preprocessed.csv'
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')

# Parse datetime
df['datetime'] = pd.to_datetime(df['datetime'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
df = df.dropna(subset=['datetime']).copy()

# Parse user_react from string back to list (if saved as string)
def parse_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str) and x.startswith('['):
        try:
            return eval(x)
        except Exception:
            return []
    return []

df['user_react'] = df['user_react'].apply(parse_list)
if 'tag' in df.columns:
    df['tag'] = df['tag'].apply(parse_list)

print(f'Posts: {len(df):,}')
print(f'Unique users: {df["user_id"].nunique():,}')
print(f'Sockpuppet users: {df[df["is_sockpuppet"]==1]["user_id"].nunique():,}')
print(f'Normal users: {df[df["is_sockpuppet"]==0]["user_id"].nunique():,}')

Posts: 34,960
Unique users: 4,895
Sockpuppet users: 959
Normal users: 3,936


## 2. Build user-level base table

One row per user with the label and group_id. All feature extraction targets this table.

In [3]:
# Aggregate to user level
user_label = (
    df.groupby('user_id')
      .agg(
          is_sockpuppet=('is_sockpuppet', 'max'),
          sockpuppet_group_id=('sockpuppet_group_id', 'max'),
          n_posts=('messages', 'count'),
      )
      .reset_index()
)

# Group IDs: -1 means normal user (no group). For GroupKFold, give each normal user
# their own unique group so they don't get accidentally bundled.
max_g = user_label['sockpuppet_group_id'].max()
is_normal = user_label['sockpuppet_group_id'] == -1
user_label.loc[is_normal, 'sockpuppet_group_id'] = (
    np.arange(is_normal.sum()) + max_g + 1
)

print(f'Users: {len(user_label):,}')
print(user_label['is_sockpuppet'].value_counts())
print(f'Distinct groups: {user_label["sockpuppet_group_id"].nunique():,}')

Users: 4,895
is_sockpuppet
0    3936
1     959
Name: count, dtype: int64
Distinct groups: 4,259


## 3. Feature Set A — Karin-style features, aggregated per user

Karin's features were originally pair-level. To make them user-level, we adapt the *underlying signals* she used:
- **Forum diversity** (how spread out across forums/tags) — direct user-level analog of forum entropy.
- **Posting hour distribution** (24-bin histogram statistics) — basis for her Time-Aware Graph.
- **Self unique-words consistency** — how repetitive the user's own vocabulary is across their posts.
- **TF-IDF self-similarity** — mean cosine similarity between the user's own posts.
- **Sockpuppet-play timing** — whether the user posts in suspicious tight time windows (rapid replies).

In [4]:
def extract_karin_style_features(df, user_label):
    records = []
    for uid in user_label['user_id']:
        u = df[df['user_id'] == uid].sort_values('datetime')
        texts = u['messages'].fillna('').tolist()
        times = u['datetime'].tolist()
        threads = u['thread_id'].tolist()

        # Forum/tag diversity (Karin's forum entropy at user level)
        if 'tag' in u.columns:
            all_tags = [t for lst in u['tag'] for t in (lst if isinstance(lst, list) else [])]
            tag_counts = Counter(all_tags)
            total = sum(tag_counts.values())
            if total > 0:
                probs = np.array([c/total for c in tag_counts.values()])
                tag_entropy = float(-np.sum(probs * np.log2(probs + 1e-10)))
            else:
                tag_entropy = 0.0
            n_distinct_tags = len(tag_counts)
        else:
            tag_entropy, n_distinct_tags = 0.0, 0

        # Posting hour distribution (basis of Karin's TAG)
        hours = [t.hour for t in times if pd.notna(t)]
        if hours:
            hour_hist = np.bincount(hours, minlength=24) / len(hours)
            hour_entropy = float(-np.sum(hour_hist * np.log2(hour_hist + 1e-10)))
            active_hours = int(np.sum(hour_hist > 0))
        else:
            hour_entropy, active_hours = 0.0, 0

        # Self-vocabulary repetition (Karin's unique-words similarity, applied internally)
        if len(texts) >= 2:
            try:
                vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4),
                                       min_df=1, max_features=2000)
                X = vec.fit_transform(texts)
                sims = cosine_similarity(X)
                # Mean of off-diagonal (self-similarity across own posts)
                np.fill_diagonal(sims, np.nan)
                mean_self_sim = float(np.nanmean(sims))
            except Exception:
                mean_self_sim = 0.0
        else:
            mean_self_sim = 0.0

        # Sockpuppet-play timing: fraction of posts that reply within 60 seconds
        # of the previous post in the SAME thread by a different user
        fast_reply_count = 0
        for _, row in u.iterrows():
            thread = row['thread_id']
            t = row['datetime']
            prev = df[(df['thread_id']==thread) & (df['datetime']<t) & (df['user_id']!=uid)]
            if len(prev) > 0:
                gap = (t - prev['datetime'].max()).total_seconds()
                if 0 < gap <= 60:
                    fast_reply_count += 1
        fast_reply_ratio = fast_reply_count / max(1, len(u))

        records.append({
            'user_id': uid,
            'k_tag_entropy': tag_entropy,
            'k_n_distinct_tags': n_distinct_tags,
            'k_hour_entropy': hour_entropy,
            'k_active_hours': active_hours,
            'k_mean_self_sim': mean_self_sim,
            'k_fast_reply_ratio': fast_reply_ratio,
        })
    return pd.DataFrame(records)

print('Extracting Karin-style features...')
X_karin = extract_karin_style_features(df, user_label)
print(X_karin.shape, list(X_karin.columns))

Extracting Karin-style features...
(4895, 7) ['user_id', 'k_tag_entropy', 'k_n_distinct_tags', 'k_hour_entropy', 'k_active_hours', 'k_mean_self_sim', 'k_fast_reply_ratio']


## 4. Feature Set B — Lindsey-style behavioral features per user

User-level behavioral fingerprint:
- **Activity volume**: total posts, distinct threads, posts per thread
- **Temporal rhythm**: mean inter-post interval, std of intervals, burst count, weekday/weekend ratio
- **Content statistics**: mean post length, length std, mean word count
- **Engagement**: total reactions received, mean reactions per post, fraction of posts that got reactions
- **Thread role**: fraction of threads where user is the initiator (num_0==0 and num_1==0)

In [5]:
def extract_lindsey_style_features(df, user_label):
    records = []
    for uid in user_label['user_id']:
        u = df[df['user_id'] == uid].sort_values('datetime')
        times = u['datetime'].tolist()
        lengths = u['length'].fillna(0).astype(int).tolist()

        n_posts = len(u)
        n_threads = u['thread_id'].nunique()
        posts_per_thread = n_posts / max(1, n_threads)

        # Temporal
        if len(times) >= 2:
            intervals = np.diff(sorted([t.timestamp() for t in times if pd.notna(t)]))
            mean_interval = float(np.mean(intervals))
            std_interval = float(np.std(intervals))
            # bursts: number of consecutive posts within 60 seconds
            burst_count = int(np.sum(intervals < 60))
        else:
            mean_interval = std_interval = burst_count = 0.0

        weekdays = [t.weekday() for t in times if pd.notna(t)]
        weekend_ratio = float(np.mean([w >= 5 for w in weekdays])) if weekdays else 0.0

        # Content statistics
        mean_len = float(np.mean(lengths)) if lengths else 0.0
        std_len = float(np.std(lengths)) if len(lengths) > 1 else 0.0

        # Engagement
        n_reacts = [len(r) if isinstance(r, list) else 0 for r in u['user_react']]
        total_reacts = int(sum(n_reacts))
        mean_reacts = float(np.mean(n_reacts)) if n_reacts else 0.0
        frac_with_reacts = float(np.mean([r > 0 for r in n_reacts])) if n_reacts else 0.0

        # Thread role — initiator if num_0 == 0 and num_1 == 0 in any thread
        if 'num_0' in u.columns and 'num_1' in u.columns:
            init_threads = u[(u['num_0']==0) & (u['num_1']==0)]['thread_id'].nunique()
        else:
            init_threads = 0
        frac_initiated = init_threads / max(1, n_threads)

        records.append({
            'user_id': uid,
            'l_n_posts': n_posts,
            'l_n_threads': n_threads,
            'l_posts_per_thread': posts_per_thread,
            'l_mean_interval': mean_interval,
            'l_std_interval': std_interval,
            'l_burst_count': burst_count,
            'l_weekend_ratio': weekend_ratio,
            'l_mean_len': mean_len,
            'l_std_len': std_len,
            'l_total_reacts': total_reacts,
            'l_mean_reacts': mean_reacts,
            'l_frac_with_reacts': frac_with_reacts,
            'l_frac_initiated': frac_initiated,
        })
    return pd.DataFrame(records)

print('Extracting Lindsey-style features...')
X_lindsey = extract_lindsey_style_features(df, user_label)
print(X_lindsey.shape, list(X_lindsey.columns))

Extracting Lindsey-style features...
(4895, 14) ['user_id', 'l_n_posts', 'l_n_threads', 'l_posts_per_thread', 'l_mean_interval', 'l_std_interval', 'l_burst_count', 'l_weekend_ratio', 'l_mean_len', 'l_std_len', 'l_total_reacts', 'l_mean_reacts', 'l_frac_with_reacts', 'l_frac_initiated']


## 5. Train/test split with group leakage prevention

Stratify by label, group by `sockpuppet_group_id`.

In [6]:
from sklearn.model_selection import GroupShuffleSplit

y = user_label['is_sockpuppet'].values
groups = user_label['sockpuppet_group_id'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(user_label, y, groups))

print(f'Train: {len(train_idx)} users ({y[train_idx].sum()} sockpuppet)')
print(f'Test:  {len(test_idx)} users ({y[test_idx].sum()} sockpuppet)')
print(f'Group overlap between train/test: '
      f'{len(set(groups[train_idx]) & set(groups[test_idx]))} (should be 0)')

Train: 3814 users (645 sockpuppet)
Test:  1081 users (314 sockpuppet)
Group overlap between train/test: 0 (should be 0)


## 6. Run the comparison

Same classifiers as Karin (DT, RF, DNN-replaced-by-LR, NB, SVM) so the comparison stays fair.

In [7]:
def run_experiment(X, y, name, train_idx, test_idx, groups_train, feature_names=None):
    print(f'\n{"="*70}')
    print(f'  Feature set: {name}  ({X.shape[1]} features, {X.shape[0]} users)')
    print(f'{"="*70}')

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    models = {
        'DT':  DecisionTreeClassifier(random_state=42),
        'RF':  RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced'),
        'NB':  GaussianNB(),
        'SVM': SVC(kernel='linear', random_state=42, class_weight='balanced', probability=True),
        'LR':  LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    }

    results = []
    for mname, m in models.items():
        Xtr = X_train_s if mname in ('SVM','LR','NB') else X_train
        Xte = X_test_s  if mname in ('SVM','LR','NB') else X_test

        m.fit(Xtr, y_train)
        y_pred = m.predict(Xte)
        y_prob = m.predict_proba(Xte)[:,1] if hasattr(m,'predict_proba') else None

        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        acc  = accuracy_score(y_test, y_pred)
        auc  = roc_auc_score(y_test, y_prob) if y_prob is not None else 0.0

        gkf = GroupKFold(n_splits=5)
        cv = cross_val_score(m, Xtr, y_train, cv=gkf, scoring='f1', groups=groups_train)

        results.append({
            'feature_set': name, 'model': mname,
            'precision': prec, 'recall': rec, 'f1': f1, 'acc': acc, 'auc': auc,
            'cv_f1_mean': cv.mean(), 'cv_f1_std': cv.std(),
        })
        print(f'  {mname:4s}  P={prec:.3f}  R={rec:.3f}  F1={f1:.3f}  AUC={auc:.3f}  CV-F1={cv.mean():.3f}±{cv.std():.3f}')
    return results

# Build matrices aligned to user_label order
X_karin_mat   = X_karin.set_index('user_id').loc[user_label['user_id']].values
X_lindsey_mat = X_lindsey.set_index('user_id').loc[user_label['user_id']].values
X_combined_mat = np.hstack([X_karin_mat, X_lindsey_mat])

groups_train = groups[train_idx]

all_results = []
all_results += run_experiment(X_karin_mat,   y, 'A: Karin-style (user-agg)',  train_idx, test_idx, groups_train)
all_results += run_experiment(X_lindsey_mat, y, 'B: Lindsey behavioral',      train_idx, test_idx, groups_train)
all_results += run_experiment(X_combined_mat, y, 'C: Combined (A+B)',          train_idx, test_idx, groups_train)

results_df = pd.DataFrame(all_results)
results_df.to_csv('user_level_comparison.csv', index=False)
print('\nSaved: user_level_comparison.csv')


  Feature set: A: Karin-style (user-agg)  (6 features, 4895 users)
  DT    P=0.442  R=0.182  F1=0.257  AUC=0.533  CV-F1=0.256±0.054
  RF    P=0.500  R=0.067  F1=0.118  AUC=0.536  CV-F1=0.126±0.021
  NB    P=0.554  R=0.178  F1=0.270  AUC=0.611  CV-F1=0.319±0.069
  SVM   P=0.400  R=0.325  F1=0.359  AUC=0.580  CV-F1=0.359±0.071
  LR    P=0.439  R=0.401  F1=0.419  AUC=0.624  CV-F1=0.366±0.044

  Feature set: B: Lindsey behavioral  (13 features, 4895 users)
  DT    P=0.439  R=0.220  F1=0.293  AUC=0.557  CV-F1=0.248±0.050
  RF    P=0.485  R=0.153  F1=0.232  AUC=0.615  CV-F1=0.169±0.056
  NB    P=0.508  R=0.099  F1=0.165  AUC=0.675  CV-F1=0.217±0.047
  SVM   P=0.424  R=0.277  F1=0.335  AUC=0.635  CV-F1=0.330±0.061
  LR    P=0.458  R=0.468  F1=0.463  AUC=0.672  CV-F1=0.342±0.057

  Feature set: C: Combined (A+B)  (19 features, 4895 users)
  DT    P=0.497  R=0.248  F1=0.331  AUC=0.579  CV-F1=0.241±0.040
  RF    P=0.506  R=0.134  F1=0.212  AUC=0.638  CV-F1=0.170±0.076
  NB    P=0.568  R=0.172  

## 7. Summary

Best model per feature set:

In [8]:
print(results_df.loc[results_df.groupby('feature_set')['f1'].idxmax()]
        [['feature_set','model','precision','recall','f1','auc']]
        .to_string(index=False))

              feature_set model  precision   recall       f1      auc
A: Karin-style (user-agg)    LR   0.439024 0.401274 0.419301 0.624214
    B: Lindsey behavioral    LR   0.457944 0.468153 0.462992 0.672448
        C: Combined (A+B)    LR   0.449814 0.385350 0.415094 0.658868
